# 06 · 频繁模式挖掘

**输入文件**：`../data/train.csv`  
**输出文件**：
- `../outputs/figures/06_*.png`
- `../outputs/tables/06_frequent_itemsets.csv`
- `../outputs/tables/06_association_rules.csv`  

**研究问题**：正例客户（target=1）中，哪些特征的取值区间倾向于同时出现？  
**随机种子**：42  
**方法**：等频分箱（3档 low/mid/high）→ 构建事务数据库 → FP-Growth → 关联规则分析  

> **注意**：Santander 的 200 个特征为匿名数值型，特征间相关性极弱（设计上独立），  
> 因此 lift 通常偏低（约 1.0~1.1 量级）。本分析使用 min_support=0.05、min_lift=1.01，  
> 重点关注正例子集内的相对高频共现模式，而非强依赖规则。

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

SEED = 42
np.random.seed(SEED)

FIG_DIR = Path('../outputs/figures')
TAB_DIR = Path('../outputs/tables')
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

## 1. 读取数据 & 分层划分（80/20）

In [2]:
train = pd.read_csv('../data/train.csv')
feature_cols = [c for c in train.columns if c.startswith('var_')]

X = train[feature_cols]
y = train['target']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'训练集: {X_train.shape}  正例数: {y_train.sum()} ({y_train.mean():.2%})')
print(f'正例子集 (target=1): {(y_train==1).sum()} 条')

训练集: (160000, 200)  正例数: 16078 (10.05%)
正例子集 (target=1): 16078 条


## 2. 特征选择：取区分度最高的 Top 30 特征

> 200 个特征全量做 FP-Growth 会产生天文数字的项集且多数无意义。  
> 只取与 target 均值差异最大的 Top 30 特征，聚焦有预测信号的规则。

In [3]:
mean_pos = X_train[y_train == 1].mean()
mean_neg = X_train[y_train == 0].mean()
mean_diff = (mean_pos - mean_neg).abs().sort_values(ascending=False)
top_feats = mean_diff.head(30).index.tolist()

print(f'选用 Top 30 特征：{top_feats[:5]} ...')

选用 Top 30 特征：['var_139', 'var_76', 'var_149', 'var_45', 'var_21'] ...


## 3. 等频分箱（3档：low/mid/high）& 构建事务数据库

In [4]:
def discretize_features(df, feat_cols, n_bins=3, labels=['low', 'mid', 'high']):
    df_disc = df[feat_cols].copy()
    for col in feat_cols:
        df_disc[col] = pd.qcut(df[col], q=n_bins, labels=labels, duplicates='drop').astype(str)
    return df_disc

# 只对正例子集做挖掘
X_pos = X_train[y_train == 1].reset_index(drop=True)
pos_disc = discretize_features(X_pos, top_feats)

print(f'正例子集形状: {pos_disc.shape}')
pos_disc.head(3)

正例子集形状: (16078, 30)


,var_139,var_76,var_149,var_45,var_21,var_174,var_184,var_80,var_40,var_90,...,var_199,var_86,var_74,var_165,var_190,var_13,var_173,var_110,var_49,var_123
0,high,mid,mid,mid,mid,high,mid,mid,mid,high,...,mid,high,low,mid,mid,mid,high,mid,high,high
1,mid,mid,high,high,low,high,mid,low,high,low,...,high,low,high,mid,mid,high,high,high,mid,high
2,low,high,high,mid,low,low,high,mid,low,high,...,high,mid,low,low,high,low,high,high,high,mid


In [5]:
# 每行转为 item 列表，格式：["var_X=high", "var_Y=low", ...]
transactions = [
    [f'{col}={val}' for col, val in row.items()]
    for _, row in pos_disc.iterrows()
]

te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_te = pd.DataFrame(te_array, columns=te.columns_)

print(f'事务数据库形状: {df_te.shape}  |  item 总数: {len(te.columns_)}')

事务数据库形状: (16078, 90)  |  item 总数: 90


## 4. FP-Growth 挖掘频繁项集

> Santander 特征设计上近似独立（匿名化处理），两特征间 lift 通常在 1.0~1.1 之间。  
> 使用 min_support=0.05 确保能挖掘到 2-itemset，min_lift=1.01 筛选相对正向关联规则。

In [6]:
MIN_SUPPORT = 0.05

frequent_items = fpgrowth(df_te, min_support=MIN_SUPPORT, use_colnames=True)
frequent_items['itemset_size'] = frequent_items['itemsets'].apply(len)
frequent_items_sorted = frequent_items.sort_values(['itemset_size', 'support'], ascending=[False, False])

size_counts = frequent_items['itemset_size'].value_counts().sort_index()
print(f'频繁项集总数 (min_support={MIN_SUPPORT}): {len(frequent_items)}')
print(f'按项集大小分布：\n{size_counts.to_string()}')

频繁项集总数 (min_support=0.05): 4005
按项集大小分布：
itemset_size
1      90
2    3915


In [7]:
# 仅展示 2-itemset（便于解释）
fi2 = frequent_items[frequent_items['itemset_size'] == 2].sort_values('support', ascending=False)
print(f'\n2-itemset 支持度 Top 20：')
fi2_show = fi2.head(20).copy()
fi2_show['items'] = fi2_show['itemsets'].apply(lambda s: ' & '.join(sorted(s)))
fi2_show[['items', 'support']].head(20)


2-itemset 支持度 Top 20：


,items,support
1704,var_13=low & var_80=mid,0.117365
193,var_172=mid & var_174=high,0.116930
3693,var_67=mid & var_70=mid,0.116619
1617,var_107=high & var_110=high,0.116557
2433,var_199=high & var_76=high,0.116432
363,var_123=high & var_172=low,0.116370
689,var_67=high & var_80=low,0.116246
2560,var_40=high & var_80=high,0.116121
1533,var_40=mid & var_86=low,0.116059
2512,var_173=low & var_40=high,0.115997


In [8]:
# 保存全量频繁项集
save_df = frequent_items_sorted.copy()
save_df['itemsets'] = save_df['itemsets'].apply(lambda s: ', '.join(sorted(s)))
save_df.to_csv(TAB_DIR / '06_frequent_itemsets.csv', index=False)
print('saved: 06_frequent_itemsets.csv')

saved: 06_frequent_itemsets.csv


## 5. 关联规则分析

In [9]:
MIN_LIFT = 1.01

rules = association_rules(frequent_items, metric='lift', min_threshold=MIN_LIFT)
rules_sorted = rules.sort_values('lift', ascending=False).reset_index(drop=True)

# 只保留 2 项集规则便于解释
rules_2item = rules_sorted[
    (rules_sorted['antecedents'].apply(len) == 1) &
    (rules_sorted['consequents'].apply(len) == 1)
].reset_index(drop=True)

print(f'关联规则总数 (min_lift={MIN_LIFT}): {len(rules_sorted)}')
print(f'二项规则数: {len(rules_2item)}')

top_rules = rules_2item.head(20).copy()
top_rules['antecedents_str'] = top_rules['antecedents'].apply(lambda s: ', '.join(sorted(s)))
top_rules['consequents_str'] = top_rules['consequents'].apply(lambda s: ', '.join(sorted(s)))
print('\nLift Top 10 二项关联规则：')
top_rules[['antecedents_str','consequents_str','support','confidence','lift']].head(10)

关联规则总数 (min_lift=1.01): 2036
二项规则数: 2036

Lift Top 10 二项关联规则：


,antecedents_str,consequents_str,support,confidence,lift
0,var_80=mid,var_13=low,0.117365,0.352184,1.056420
1,var_13=low,var_80=mid,0.117365,0.352052,1.056420
2,var_172=mid,var_174=high,0.116930,0.350812,1.052501
3,var_174=high,var_172=mid,0.116930,0.350812,1.052501
4,var_70=mid,var_67=mid,0.116619,0.349879,1.049701
5,var_67=mid,var_70=mid,0.116619,0.349879,1.049701
6,var_110=high,var_107=high,0.116557,0.349692,1.049142
7,var_107=high,var_110=high,0.116557,0.349692,1.049142
8,var_76=high,var_199=high,0.116432,0.349319,1.048022
9,var_199=high,var_76=high,0.116432,0.349319,1.048022


In [10]:
# 保存
rules_save = rules_sorted.copy()
rules_save['antecedents'] = rules_save['antecedents'].apply(lambda s: ', '.join(sorted(s)))
rules_save['consequents'] = rules_save['consequents'].apply(lambda s: ', '.join(sorted(s)))
rules_save.head(50).to_csv(TAB_DIR / '06_association_rules.csv', index=False)
print('saved: 06_association_rules.csv')

saved: 06_association_rules.csv


## 6. 可视化

In [11]:
# 支持度 Top 20 频繁 2-itemset 条形图
top20_2 = fi2.head(20).copy()
top20_2['label'] = top20_2['itemsets'].apply(lambda s: ' & '.join(sorted(s)))

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(top20_2['label'][::-1], top20_2['support'][::-1], color='#4C72B0', edgecolor='white')
ax.set_xlabel('Support（正例子集中的共现频率）')
ax.set_title('正例客户：支持度最高 Top 20 二项频繁项集')
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '06_top20_frequent_itemsets.png', bbox_inches='tight')
plt.show()
print('saved: 06_top20_frequent_itemsets.png')

saved: 06_top20_frequent_itemsets.png


In [12]:
# 关联规则散点图：x=support, y=confidence, 颜色=lift
if len(rules_2item) > 0:
    fig, ax = plt.subplots(figsize=(9, 6))
    sc = ax.scatter(
        rules_2item['support'],
        rules_2item['confidence'],
        c=rules_2item['lift'],
        cmap='YlOrRd',
        alpha=0.6,
        s=20,
        edgecolors='none'
    )
    plt.colorbar(sc, ax=ax, label='Lift')
    ax.set_xlabel('Support')
    ax.set_ylabel('Confidence')
    ax.set_title('关联规则散点图（x=support, y=confidence, 颜色=lift）\n正例客户 Top 30 特征子集')
    sns.despine()
    plt.tight_layout()
    plt.savefig(FIG_DIR / '06_association_rules_scatter.png', bbox_inches='tight')
    plt.show()
    print('saved: 06_association_rules_scatter.png')

saved: 06_association_rules_scatter.png


In [13]:
# Lift Top 20 关联规则条形图
if len(rules_2item) >= 5:
    top_rules['rule'] = top_rules['antecedents_str'] + ' → ' + top_rules['consequents_str']
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(top_rules['rule'][::-1], top_rules['lift'][::-1], color='#DD8452', edgecolor='white')
    ax.axvline(1.0, color='gray', linestyle='--', alpha=0.7, label='Lift=1.0 (统计独立)')
    ax.set_xlabel('Lift')
    ax.set_title('Lift 最高 Top 20 二项关联规则（正例客户子集）')
    ax.legend()
    sns.despine()
    plt.tight_layout()
    plt.savefig(FIG_DIR / '06_top20_rules_lift.png', bbox_inches='tight')
    plt.show()
    print('saved: 06_top20_rules_lift.png')

saved: 06_top20_rules_lift.png


## 7. 正负样本对比：正例 vs 负例中的高频 item 差异

In [14]:
# 负例子集同样离散化，对比单 item 支持度
X_neg = X_train[y_train == 0].reset_index(drop=True)
neg_disc = discretize_features(X_neg, top_feats)

# 计算正例和负例中每个 item 的支持度
pos_support = {}
neg_support = {}
for col in top_feats:
    for label in ['low', 'mid', 'high']:
        item = f'{col}={label}'
        pos_support[item] = (pos_disc[col] == label).mean()
        neg_support[item] = (neg_disc[col] == label).mean()

support_df = pd.DataFrame({'pos_support': pos_support, 'neg_support': neg_support})
support_df['diff'] = support_df['pos_support'] - support_df['neg_support']
support_df = support_df.sort_values('diff', ascending=False)

# 画正例中更高频 vs 负例中更高频的 Top 15 item
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_pos = support_df.head(15)
axes[0].barh(top_pos.index[::-1], top_pos['diff'][::-1], color='#DD8452', edgecolor='white')
axes[0].set_xlabel('正例支持度 - 负例支持度')
axes[0].set_title('正例中更高频的特征取值 Top 15')

top_neg = support_df.tail(15)
axes[1].barh(top_neg.index, top_neg['diff'], color='#4C72B0', edgecolor='white')
axes[1].set_xlabel('正例支持度 - 负例支持度')
axes[1].set_title('负例中更高频的特征取值 Top 15')

sns.despine()
plt.suptitle('正例 vs 负例：各特征取值区间支持度差异', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / '06_pos_neg_support_diff.png', bbox_inches='tight')
plt.show()
print('saved: 06_pos_neg_support_diff.png')

saved: 06_pos_neg_support_diff.png


## 8. 解读 Top 规则

In [15]:
print('=== Top 5 关联规则解读 ===')
for i, row in rules_2item.head(5).iterrows():
    ant = ', '.join(sorted(row['antecedents']))
    con = ', '.join(sorted(row['consequents']))
    print(f'\n规则 {i+1}: [{ant}] → [{con}]')
    print(f'  Support={row["support"]:.4f}  Confidence={row["confidence"]:.4f}  Lift={row["lift"]:.4f}')
    print(f'  解读：在正例客户中，出现 {ant} 时，{con} 同时出现的频率')
    print(f'        是随机期望的 {row["lift"]:.3f} 倍（超出期望 {(row["lift"]-1)*100:.1f}%）')

=== Top 5 关联规则解读 ===

规则 1: [var_80=mid] → [var_13=low]
  Support=0.1174  Confidence=0.3522  Lift=1.0564
  解读：在正例客户中，出现 var_80=mid 时，var_13=low 同时出现的频率
        是随机期望的 1.056 倍（超出期望 5.6%）

规则 2: [var_13=low] → [var_80=mid]
  Support=0.1174  Confidence=0.3521  Lift=1.0564
  解读：在正例客户中，出现 var_13=low 时，var_80=mid 同时出现的频率
        是随机期望的 1.056 倍（超出期望 5.6%）

规则 3: [var_172=mid] → [var_174=high]
  Support=0.1169  Confidence=0.3508  Lift=1.0525
  解读：在正例客户中，出现 var_172=mid 时，var_174=high 同时出现的频率
        是随机期望的 1.053 倍（超出期望 5.3%）

规则 4: [var_174=high] → [var_172=mid]
  Support=0.1169  Confidence=0.3508  Lift=1.0525
  解读：在正例客户中，出现 var_174=high 时，var_172=mid 同时出现的频率
        是随机期望的 1.053 倍（超出期望 5.3%）

规则 5: [var_70=mid] → [var_67=mid]
  Support=0.1166  Confidence=0.3499  Lift=1.0497
  解读：在正例客户中，出现 var_70=mid 时，var_67=mid 同时出现的频率
        是随机期望的 1.050 倍（超出期望 5.0%）


## 小结

- 只对正例客户（target=1）的 Top 30 高区分度特征做频繁模式挖掘
- **Santander 特征匿名化设计上近似独立**，特征间 lift 普遍在 1.0~1.1 量级，这是数据集本身的特性而非分析局限
- 频繁项集和关联规则分析显示，正例客户中存在若干特征取值区间的相对高频共现，但关联强度较弱
- 更有价值的发现来自**正例 vs 负例支持度差异图**：清晰显示哪些特征取值区间（如某特征的 high 档）在正例中明显更常见，这与 EDA 中的 KDE 分布分析结论相互印证
- 由于特征匿名，规则的业务解释性受限，但统计意义上的共现模式对报告的定性分析有支撑作用